# Attention Shape Reasoning: MHA, KV Cache and GQA

**Roadmap 1.1 / scope:** Learn to reason about Batch, Heads, Query tokens, Key tokens and Head dimension. This is a CPU-only teaching notebook requiring PyTorch 2.x; no model downloads, no performance benchmarks. Run cells from top to bottom.

**Task:** Predict each output shape before revealing the stored results.

In [1]:
import torch
import math
print("Using PyTorch on CPU")

Using PyTorch on CPU


## 1. Split Q/K/V into heads

Input `(B,T,D)=(2,6,12)`, `H=3`, so `Dh=4`. A projection changes features, not batch or token count. Then reshape `(B,T,H,Dh)` and permute into `(B,H,T,Dh)`.

In [2]:
torch.manual_seed(7)
B, T, D, H = 2, 6, 12, 3
Dh = D // H
x = torch.randn(B, T, D)
q_proj = torch.nn.Linear(D, D, bias=False)
k_proj = torch.nn.Linear(D, D, bias=False)
v_proj = torch.nn.Linear(D, D, bias=False)
def split_heads(t):
    return t.reshape(B, T, H, Dh).permute(0, 2, 1, 3)
q = split_heads(q_proj(x))
k = split_heads(k_proj(x))
v = split_heads(v_proj(x))
print("input:", tuple(x.shape))
print("Q/K/V:", tuple(q.shape))
print("weight:", tuple(q_proj.weight.shape))

input: (2, 6, 12)
Q/K/V: (2, 3, 6, 4)
weight: (12, 12)


## 2. Batched matrix multiplication and softmax

Scores are `(B,H,Tq,Tk)`, context is `(B,H,Tq,Dh)`. The softmax is over **keys**. This pedagogical example is unmasked bidirectional attention, *not* a complete causal decoder.

In [3]:
scores = (q @ k.transpose(-2, -1)) / math.sqrt(Dh)
weights = scores.softmax(dim=-1)
context = weights @ v
merged = context.permute(0, 2, 1, 3).contiguous().view(B, T, D)
print("scores:", tuple(scores.shape))
print("weights:", tuple(weights.shape))
print("context:", tuple(context.shape))
print("merged:", tuple(merged.shape))
print("per-query weights sum to one:", torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1))))

scores: (2, 3, 6, 6)
weights: (2, 3, 6, 6)
context: (2, 3, 6, 4)
merged: (2, 6, 12)
per-query weights sum to one: True


## 3. Prefill vs one decode step

When the decoder uses a KV cache, query length can be 1 while key length includes history. We compare the number of logical score entries, **not actual fused-kernel VRAM allocation**.

In [4]:
B2, H2, T2, Dh2 = 2, 4, 12, 16
prefill_q = torch.zeros(B2, H2, T2, Dh2)
prefill_k = torch.zeros(B2, H2, T2, Dh2)
prefill_scores = prefill_q @ prefill_k.transpose(-2, -1)
decode_q = torch.zeros(B2, H2, 1, Dh2)
decode_k = torch.zeros(B2, H2, T2 + 1, Dh2)
decode_v = torch.zeros_like(decode_k)
decode_scores = decode_q @ decode_k.transpose(-2, -1)
decode_context = decode_scores.softmax(-1) @ decode_v
print("prefill:", tuple(prefill_scores.shape), "numel:", prefill_scores.numel())
print("decode:", tuple(decode_scores.shape), "numel:", decode_scores.numel())
print("context:", tuple(decode_context.shape))

prefill: (2, 4, 12, 12) numel: 1152
decode: (2, 4, 1, 13) numel: 104
context: (2, 4, 1, 16)


## 4. Silent bug: correct permute, wrong reshape

Same shape does **not** imply the same meaning. From `(B,T,H,Dh)` to `(B,H,T,Dh)`, swapping axes preserves the head-to-token mapping; mere reshape can mix them.

In [5]:
original = torch.arange(8).reshape(1, 2, 2, 2)
good = original.permute(0, 2, 1, 3)
bad = original.reshape(1, 2, 2, 2)
print("head 0 correct:", good[0, 0].tolist())
print("head 0 wrong:", bad[0, 0].tolist())
print("same shape:", tuple(good.shape) == tuple(bad.shape))

head 0 correct: [[0, 1], [4, 5]]
head 0 wrong: [[0, 1], [2, 3]]
same shape: True


## 5. GQA: group Query heads without duplicating stored KV

`Hq=8`, `Hkv=2` and `group=4`. Organize Q as `(B,Hkv,group,Tq,Dh)`; introduce a singleton group axis for K/V and let matmul broadcast across groups. For **this eager PyTorch demo**, avoid explicitly repeating cached K/V during grouped computation. A production native GQA kernel can be more efficient.

In [6]:
B3, Hq, Hkv, Tq, Tk, Dh3 = 2, 8, 2, 1, 16, 32
G = Hq // Hkv
torch.manual_seed(7)
q3 = torch.randn(B3, Hq, Tq, Dh3)
k3 = torch.randn(B3, Hkv, Tk, Dh3)
v3 = torch.randn(B3, Hkv, Tk, Dh3)
q_grouped = q3.reshape(B3, Hkv, G, Tq, Dh3)
k_grouped = k3.unsqueeze(2)
v_grouped = v3.unsqueeze(2)
gqa_scores = q_grouped @ k_grouped.transpose(-2, -1)
gqa_context = (gqa_scores.softmax(-1) @ v_grouped).reshape(B3, Hq, Tq, Dh3)
# Baseline is intentionally materialized only to validate numerical equivalence.
k_repeated = k3.repeat_interleave(G, dim=1)
v_repeated = v3.repeat_interleave(G, dim=1)
baseline = (q3 @ k_repeated.transpose(-2, -1)).softmax(-1) @ v_repeated
print("grouped scores:", tuple(gqa_scores.shape))
print("GQA context:", tuple(gqa_context.shape))
print("matches repeated baseline:", torch.allclose(gqa_context, baseline))
print("K cache bytes / repeated K bytes:", k3.numel()*k3.element_size(), "/", k_repeated.numel()*k_repeated.element_size())

grouped scores: (2, 2, 4, 1, 16)
GQA context: (2, 8, 1, 32)
matches repeated baseline: True
K cache bytes / repeated K bytes: 8192 / 32768


## Reflect, then extend

- Why does a query length of one imply rectangular scores in decode?
- Why do two attention tensors with identical shapes sometimes produce different results?
- Why does GQA reduce cache storage but not necessarily total compute by the same factor?
- How would a causal mask change the valid keys for each query in prefill?

**Evidence boundary:** All outputs above are deterministic CPU exercises. They do not demonstrate GPU speedups, VRAM peaks, benchmarked FlashAttention behavior or independent L3 implementation.

Reference notes: [Tensor Fundamentals for LLM](../notes/01-tensor-fundamentals-for-llm.md).